In [41]:
import pandas as pd
import numpy as np

In [103]:
#Central Path variable
RAW_PATH="../data/raw/online_retail_II.xlsx"

In [18]:
# Reload the raw data 
sheet_2009_2010 = pd.read_excel(RAW_PATH, sheet_name = "Year 2009-2010")
sheet_2010_2011 = pd.read_excel(RAW_PATH, sheet_name ="Year 2010-2011")
sheet_2009_2010["SourceSheet"] = "2009-2010"
sheet_2010_2011["SourceSheet"] = "2010-2011"

In [33]:
#Stack both sheets into one dataframe
df = pd.concat([sheet_2009_2010, sheet_2010_2011], ignore_index = True)
print(len(sheet_2009_2010))
print(len(sheet_2010_2011))
print(len(df))

525461
541910
1067371


In [39]:
#Confirm both sheets share identical column names
print(list(sheet_2009_2010.columns)==list(sheet_2010_2011.columns))

True


In [79]:
#Transaction type classification rules, built directly from profiling findings.
is_bad_debt = (df["StockCode"]=="B")
is_manual_entry = (df["StockCode"]=="M")
is_discount = (df["StockCode"]=="D")
is_postage = (df["StockCode"]=="POST") | (df["StockCode"]=="DOT")
is_carriage = (df["StockCode"]=="C2")
is_stock_write_off = (df["Quantity"]<0) &(df["Price"]==0)

In [81]:
# Combine all classification rules into ONE TransactionType column via np.select.
# Checked in order, first match wins; anything matching none of the rules defaults to "Sale".
# NOTE: deliberately does NOT include a cancellation rule here cancellation status is
# tracked independently in IsCancelled below, since a row can be both e.g. a cancelled
# Manual Entry, and cramming both facts into one label would hide that overlap
conditions =[
    is_bad_debt,
    is_manual_entry,
    is_discount,
    is_postage,
    is_carriage,
    is_stock_write_off
]
labels =[
    "Bad Debt",
    "Manual Entry",
    "Discount",
    "Postage",
    "Carriage",
    "Stock Write-off"
]
df["TransactionType"] = np.select(conditions, labels, default ="Sale")

In [83]:
#Independent cancellation flag True if Invoice starts with 'C'.
df["IsCancelled"] = (df["Invoice"].astype(str).str[0] == "C")

In [85]:
#Sanity check
df["TransactionType"].value_counts()

TransactionType
Sale               1058460
Postage               3568
Stock Write-off       3457
Manual Entry          1421
Carriage               282
Discount               177
Bad Debt                 6
Name: count, dtype: int64

In [87]:
# Debug check that caught the original bug confirms 172 of 177 Discount rows also sit
# on C prefixed (cancelled) invoices, which is why they were being mislabeled 'Cancellation'
# before TransactionType and IsCancelled were split into separate columns
df[df["StockCode"] == "D"]["Invoice"].astype(str).str[0].value_counts()

Invoice
C    172
5      5
Name: count, dtype: int64

In [89]:
# Confirm IsCancelled reconciles to the full C-invoice count from profiling (19,494) 
#now correct after removing the StockCode!='M' exception that was hiding 500 rows
df["IsCancelled"].value_counts()

IsCancelled
False    1047877
True       19494
Name: count, dtype: int64

In [91]:
#Full dataframe preview
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,TransactionType,IsCancelled
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010,Sale,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,Sale,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,Sale,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010,Sale,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010,Sale,False
...,...,...,...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,2010-2011,Sale,False
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,2010-2011,Sale,False
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,2010-2011,Sale,False
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,2010-2011,Sale,False


In [93]:
# Build a whitespace stripped copy of Description (kept as a NEW column, not overwriting
# the original, so we can always trace back to the raw text if needed).
df["Description_clean"] =df["Description"].str.strip()

In [99]:
# Quantify how many rows actually had a whitespace difference 217,421 rows
(df["Description"] !=df["Description_clean"]).sum()

217421

In [101]:
#Known stock note / operational annotation vocabulary (from profiling investigation)
note_words = ["check", "lost", "missing", "smashed", "short", "damaged", "found", "wrong", "?", "thrown away", "faulty", "update"]